# ASPIRE Quickstart Notebook

This walkthrough shows how to load a pretrained ASPIRE model, run a single-record prediction, and (soon) score an entire dataset.

In [1]:
%pip install .

Processing /Users/edddddy/Documents/ASPIRE/open-source/aspire
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for aspire: filename=aspire-0.1-py3-none-any.whl size=22588 sha256=1c44f530ee0cc6212115574cab142600eeb1de8fc57ad5e28d5b453dcaa757fd
  Stored in directory: /private/var/folders/79/h88rl61936g_75bvljbb2d200000gn/T/pip-ephem-wheel-cache-8x5g7b0k/wheels/81/4c/2a/ac791919f82c85ffd6e3c8c889f123a212e772ab4e6109e4e1
Successfully built aspire
  Attempting uninstall: aspire
    Found existing installation: aspire 0.1
    Uninstalling aspire-0.1:
      Successfully uninstalled aspire-0.1

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip show aspire

Name: aspire
Version: 0.1
Summary: ASPIRE — Arbitrary Set-based Permutation-Invariant Reasoning Engine
Home-page: https://github.com/placeholder
Author: 
Author-email: 
License: CC BY-NC 4.0
Location: /opt/homebrew/lib/python3.10/site-packages
Requires: huggingface-hub, numpy, pandas, scikit-learn, torch, tqdm, transformers
Required-by: 


## 1. Load the pretrained model

In [3]:
from aspire import AspireModel

model = AspireModel.from_pretrained(repo_id="checkpoints/best_model", device="cpu")
model

/opt/homebrew/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/homebrew/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


AspireModel(
  (model): ASPIREEnhanced(
    (semantic_grounding): SemanticFeatureGrounding(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.1, inplace=False)
                )
                (output): BertSe

## 2. Predict for a single record

In [4]:
from aspire.model import Feature, Example

# 1) Describe your schema once
feature_specs = [
    {"name": "age", "description": "Age in years", "dtype": "continuous", "value_range": (18, 80)},
    {"name": "sleep_hours_last_night", "description": "Total sleep duration last night (hours)", "dtype": "continuous", "value_range": (0.0, 12.0)},
    {"name": "caffeine_mg", "description": "Caffeine consumed today (mg)", "dtype": "continuous", "value_range": (0, 600)},
    {"name": "stress_level", "description": "Self-reported stress level", "dtype": "categorical", "choices": ["low", "medium", "high"]},
    {"name": "physical_activity_min", "description": "Physical activity duration today (minutes)", "dtype": "continuous", "value_range": (0, 240)},
    {"name": "work_type", "description": "Primary work type during day", "dtype": "categorical", "choices": ["desk", "manual", "shift_night"]},
    {"name": "ambient_temperature_c", "description": "Ambient temperature during activity (°C)", "dtype": "continuous", "value_range": (10, 40)},
    {"name": "hours_until_fatigue", "description": "Estimated time until notable fatigue onset (hours)", "dtype": "continuous", "value_range": (0.0, 12.0)},
]
features = [Feature(**spec) for spec in feature_specs]

# 2) Define values for a single record
record = {
    "age": 35,
    "sleep_hours_last_night": 5.5,
    "caffeine_mg": 120,
    "stress_level": "high",
    "physical_activity_min": 60,
    "work_type": "manual",
    "ambient_temperature_c": 30,
    "hours_until_fatigue": None,  # target
}

# 3) Pick which columns should be predicted
target_names = ["hours_until_fatigue"]
target_indices = [i for i, feat in enumerate(features) if feat.name in target_names]

# 4) Convert to the Example structure the model expects
values = [record.get(feat.name) for feat in features]
example = Example(
    features=features,
    values=values,
    target_indices=target_indices,
    dataset_context=(
        "A human performance dataset combining sleep, caffeine, stress, work type, "
        "activity, and environmental factors to estimate time until noticeable fatigue."
    ),
    support_examples=None,  # optionally add few-shot demonstrations later
)

predictions = model.predict(example)
predictions

[6.032322406768799]

## 3. Adding Metadata to Dataset

In [5]:
import json
import pandas as pd
from pathlib import Path
from aspire.model import Feature
from aspire.data_loader import add_metadata

# 1) Load dataset/table to a pandas dataframe
dataset_path = "datasets/synthetic_pet_wellness_programs_example.csv"
df = pd.read_csv(dataset_path)

# 2) Load metadata
metadata_path = Path("datasets/metadata_pet_wellness_programs_example.json")
metadata = json.loads(metadata_path.read_text())
metadata["description"]

'A dataset designed to evaluate and enhance the wellness of pets through various health and behavioral metrics.'

In [6]:
# 2.1) Option 1: Extract feature descriptions into List[Feature] manually
feature_desc1 = [
    Feature(
        name=item["name"],
        description=item.get("description", ""),
        dtype=("continuous" if (item.get("dtype") or item.get("type")) == "continuous" else "categorical"),
        choices=item.get("choices") or item.get("categories"),
        value_range=tuple(item.get("value_range") or item.get("range") or []) or None,
    )
    for item in metadata["features"]
]
feature_desc1

[Feature(name='pet_age', description='Age of the pet in years at the time of assessment', dtype='continuous', choices=None, value_range=(0.1, 25.0)),
 Feature(name='species', description='Biological species of the pet', dtype='categorical', choices=['Dog', 'Cat', 'Bird', 'Reptile', 'Fish', 'Small Mammal'], value_range=None),
 Feature(name='weight_kg', description='Current weight of the pet measured in kilograms', dtype='continuous', choices=None, value_range=(0.1, 50.0)),
 Feature(name='vaccination_status', description='Indicates current vaccination completeness', dtype='categorical', choices=['Up-to-date', 'Overdue', 'Incomplete', 'Not applicable'], value_range=None),
 Feature(name='last_vet_visit_days_ago', description='Number of days since the last veterinary visit', dtype='continuous', choices=None, value_range=(0.0, 365.0)),
 Feature(name='activity_level', description='Quantified activity level based on tracking device data', dtype='categorical', choices=['Low', 'Moderate', 'High'

In [7]:
# 2.2) Option 2: Extract feature descriptions into List[str] (name, dtype, choices/value_range will be inferred)
feature_desc2 = [feature.get("description", "") for feature in metadata["features"]]
feature_desc2

['Age of the pet in years at the time of assessment',
 'Biological species of the pet',
 'Current weight of the pet measured in kilograms',
 'Indicates current vaccination completeness',
 'Number of days since the last veterinary visit',
 'Quantified activity level based on tracking device data',
 'Type of diet currently followed',
 'Composite score from 0-100 assessing behavioral health',
 'Current parasite prevention method usage status',
 "Assessment of owner's adherence to care recommendations (0-10 scale)",
 'Presence and type of allergies',
 'Assigned risk category based on overall health assessment']

In [8]:
# 3) Attach metadata to the DataFrame
df_with_meta = add_metadata(
    df.copy(),
    feature_desc=feature_desc1,
    target=["health_risk_category"], # Optionally save target as metadata
    dataset_desc=metadata["description"],
)
df_with_meta.attrs

{'feature_desc': [{'name': 'pet_age',
   'description': 'Age of the pet in years at the time of assessment',
   'dtype': 'continuous',
   'choices': None,
   'value_range': (0.1, 25.0)},
  {'name': 'species',
   'description': 'Biological species of the pet',
   'dtype': 'categorical',
   'choices': ['Dog', 'Cat', 'Bird', 'Reptile', 'Fish', 'Small Mammal'],
   'value_range': None},
  {'name': 'weight_kg',
   'description': 'Current weight of the pet measured in kilograms',
   'dtype': 'continuous',
   'choices': None,
   'value_range': (0.1, 50.0)},
  {'name': 'vaccination_status',
   'description': 'Indicates current vaccination completeness',
   'dtype': 'categorical',
   'choices': ['Up-to-date', 'Overdue', 'Incomplete', 'Not applicable'],
   'value_range': None},
  {'name': 'last_vet_visit_days_ago',
   'description': 'Number of days since the last veterinary visit',
   'dtype': 'continuous',
   'choices': None,
   'value_range': (0.0, 365.0)},
  {'name': 'activity_level',
   'desc

In [9]:
# Alternative: If metadata json is pandas attrs styled, attach dict to df directly
metadata_pd_path = Path("datasets/metadata_pet_wellness_programs_example_pd.json")
metadata_pd = json.loads(metadata_pd_path.read_text())

df_with_meta = add_metadata(df.copy(), metadata=metadata_pd)
df_with_meta.attrs

{'feature_desc': [{'name': 'pet_age',
   'description': 'Age of the pet in years at the time of assessment',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.1, 25.0]},
  {'name': 'species',
   'description': 'Biological species of the pet',
   'dtype': 'categorical',
   'choices': ['Dog', 'Cat', 'Bird', 'Reptile', 'Fish', 'Small Mammal'],
   'value_range': None},
  {'name': 'weight_kg',
   'description': 'Current weight of the pet measured in kilograms',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.1, 50.0]},
  {'name': 'vaccination_status',
   'description': 'Indicates current vaccination completeness',
   'dtype': 'categorical',
   'choices': ['Up-to-date', 'Overdue', 'Incomplete', 'Not applicable'],
   'value_range': None},
  {'name': 'last_vet_visit_days_ago',
   'description': 'Number of days since the last veterinary visit',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 365.0]},
  {'name': 'activity_level',
   'desc

## 4. Predicting for Dataset

In [10]:
# Option 1: If metadata has target, predict using metadata by default
meta_predictions1 = model.predict(df_with_meta) # Uses target from metadata
meta_predictions1

[['Low risk'],
 ['Low risk'],
 ['Low risk'],
 ['Low risk'],
 ['Low risk'],
 ['Low risk'],
 ['Low risk'],
 ['Low risk'],
 ['Low risk']]

In [11]:
# Option 2: Assign target manually
target_columns = ["dietary_plan", "health_risk_category"]
meta_predictions2 = model.predict(df_with_meta, target_columns) # Explicitly define target
meta_predictions2

[['Raw', 'Low risk'],
 ['Raw', 'Low risk'],
 ['Raw', 'Low risk'],
 ['Raw', 'Low risk'],
 ['Raw', 'Low risk'],
 ['Raw', 'Low risk'],
 ['Raw', 'Low risk'],
 ['Raw', 'Low risk'],
 ['Raw', 'Low risk']]

## 5. Metadata Creation

For datasets that do not have original metadata, create a template by using create_metadata_template.

In [12]:
from aspire.data_loader import create_metadata_template

# 1) Generate a metadata template 
dataset_path = "datasets/heart-c.csv"
metadata_path = create_metadata_template(dataset_path, overwrite=True)

Please edit metadata file at  datasets/metadata_heart-c.json


After running create_metadata_template, find the generated template json at /dataset. Fill in any missing dataset description and feature description. The generated template uses inferred choices/value_rage from provided dataset, so double check if they're as intended.

In [13]:
# 2) Manually edit the metadata at printed dir.

# 3) Read the metadata json and directly attach it to dataframe
metadata = json.loads(Path(metadata_path).read_text())
df = pd.read_csv(dataset_path)

df_with_meta = add_metadata(df.copy(), metadata=metadata)

df_with_meta.attrs

{'dataset_desc': 'Dataset description here',
 'target': [],
 'feature_desc': [{'name': 'age',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'sex',
   'description': '',
   'dtype': 'categorical',
   'choices': ['female', 'male'],
   'value_range': None},
  {'name': 'cp',
   'description': '',
   'dtype': 'categorical',
   'choices': ['non_anginal', 'asympt', 'atyp_angina', 'typ_angina'],
   'value_range': None},
  {'name': 'trestbps',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 1.0]},
  {'name': 'chol',
   'description': '',
   'dtype': 'continuous',
   'choices': None,
   'value_range': [0.0, 0.9999999999999998]},
  {'name': 'fbs',
   'description': '',
   'dtype': 'categorical',
   'choices': ['f', 't'],
   'value_range': None},
  {'name': 'restecg',
   'description': '',
   'dtype': 'categorical',
   'choices': ['st_t_wave_abnormality', 'left_vent_hyper', 'normal'],
   'va

## 6. Backup: Predicting without Metadata

In [ ]:
# 1) Load your table to a pandas dataframe
dataset_path = "datasets/synthetic_pet_wellness_programs_example.csv"
df = pd.read_csv(dataset_path)

# 2) Decide which columns to predict
target_columns = ["health_risk_category"]

# 3) Run inference
predictions_list = model.predict(df, target_columns)
predictions_list

[['High risk'],
 ['Low risk'],
 ['Low risk'],
 ['High risk'],
 ['High risk'],
 ['High risk'],
 ['Low risk'],
 ['Low risk'],
 ['High risk']]